<a href="https://colab.research.google.com/github/dynastygeek/ETF_Prices/blob/main/ETFs_High_and_Low_Prediction_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ** Based on "price_date" and "fund_symbol" we will predict what can be the "low" and "high" price for the specific date **

In [ ]:
# import the libraries
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

In [ ]:
# read the dataset
df = pd.read_csv("/content/ETF prices.csv")


In [ ]:
# understanding what's in dataset
df.head()
df.columns

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['fund_symbol'].nunique()

In [ ]:
df['fund_symbol'].unique()

In [ ]:
# converting the datatypes of following columns to minimize the memoryusage
df['open'] = df['open'].astype('float32')
df['high'] = df['high'].astype('float32')
df['low'] = df['low'].astype('float32')
df['close'] = df['close'].astype('float32')
df['adj_close'] = df['adj_close'].astype('float32')
df['volume'] = df['volume'].astype('float32')
df['price_date'] = pd.to_datetime(df['price_date'])


In [ ]:
df.info()

In [ ]:
print(df['price_date'].min())
print(df['price_date'].max())

In [ ]:
df.groupby('fund_symbol')['price_date'].count().sort_values()

In [ ]:
df = df.sort_values(['fund_symbol', 'price_date'])

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
# 4. check for duplicate records in the data

df.duplicated().sum() # no duplicate record found

# df = df.drop_duplicates(subset=['fund_symbol', 'price_date'])

In [ ]:
df.duplicated(
    subset=['fund_symbol', 'price_date']
).sum()

In [ ]:
# 5. Check missing values

df.isnull().sum()

In [ ]:
# # 6. Check invalid prices
# High >= Low
# High >= Open
# High >= Close
# Low <= Open
# Low <= Close

invalid = df[
    (df['high'] < df['low']) |
    (df['high'] < df['open']) |
    (df['high'] < df['close']) |
    (df['low'] > df['open']) |
    (df['low'] > df['close'])
]

print(invalid)

In [ ]:
df = df[~df.index.isin(invalid.index)]
print(f"Number of records after removing invalid entries: {len(df)}")

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
df

In [ ]:

plt.figure(figsize=(12, 6))

plt.plot(df['price_date'], df['close'])

plt.xlabel('Date')
plt.ylabel('Close Price')
plt.title('Fund Price Over Time')

plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
fund = 'AAA'

temp = df[df['fund_symbol'] == fund]

plt.figure(figsize=(12, 6))
plt.plot(temp['price_date'], temp['close'])

plt.xlabel('Date')
plt.ylabel('Close Price')
plt.title(f'{fund} Price History')

plt.grid(True)
plt.show()

In [ ]:
# Feature engineering
# Create a target_high and low colume where the model will train for predicting next day high and low

df['target_high'] = df['high'].shift(-1)
df['target_low'] = df['low'].shift(-1)

In [ ]:
# calculating previous day low, high, close, open

df['close_lag'] = df.groupby('fund_symbol')['close'].shift(1)
df['open_lag'] = df.groupby('fund_symbol')['open'].shift(1)
df['high_lag'] = df.groupby('fund_symbol')['high'].shift(1)
df['low_lag'] = df.groupby('fund_symbol')['low'].shift(1)

In [ ]:
df

In [ ]:
for lag in [1, 2, 3, 5, 10]:
  df[f'close_lag_{lag}'] =  df.groupby('fund_symbol')['close'].shift(lag)
  df[f'open_lag_{lag}'] =  df.groupby('fund_symbol')['open'].shift(lag)
  df[f'high_lag_{lag}'] =  df.groupby('fund_symbol')['high'].shift(lag)
  df[f'low_lag_{lag}'] =  df.groupby('fund_symbol')['low'].shift(lag)

In [ ]:
df['ma_5'] = (
    df.groupby('fund_symbol')['close']
      .transform(lambda x: x.rolling(5).mean())
)

In [ ]:
for num in [10,20]:
  df[f'ma_{num}'] = (
      df.groupby('fund_symbol')['close']
        .transform(lambda x: x.rolling(num).mean())
  )

In [ ]:
df.shape

**Return = Today′s Close−Yesterday′s Close​ /Yesterday′s Close**

In [ ]:
# percentage returns in 1 and 5 days

df['return_1day'] = df.groupby('fund_symbol')['close'].pct_change(1)
df['return_5day'] = df.groupby('fund_symbol')['close'].pct_change(5)
df['return_10day'] = df.groupby('fund_symbol')['close'].pct_change(10)

In [ ]:
df

In [ ]:
df['volatility_10'] = (
    df.groupby('fund_symbol')['return_1day']
      .transform(lambda x: x.rolling(10).std())
)

In [ ]:
df['volatility_5'] = (
    df.groupby('fund_symbol')['return_5day']
      .transform(lambda x: x.rolling(5).std())
)

In [ ]:
df['price_range'] = df['high'] - df['low']

In [ ]:
df['range_pct'] = (
    (df['high'] - df['low']) / df['close']
)

In [ ]:
df


In [ ]:
df['day'] = df['price_date'].dt.day
df['day_of_week'] = df['price_date'].dt.dayofweek
df['month'] = df['price_date'].dt.month
df['quarter'] = df['price_date'].dt.quarter
df['year'] = df['price_date'].dt.year

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['fund_encoded'] = encoder.fit_transform(
    df['fund_symbol']
)

In [ ]:
df = df.dropna()

In [ ]:
df.shape

In [ ]:
fund_counts = df['fund_symbol'].value_counts()
funds_to_remove = fund_counts[fund_counts < 1000].index
df = df[~df['fund_symbol'].isin(funds_to_remove)]
print(f"Number of funds removed: {len(funds_to_remove)}")
print(f"Number of rows after filtering: {len(df)}")

In [ ]:
df.shape

In [ ]:
features = [
    'open',
    'high',
    'low',
    'close',
    'volume',

    'close_lag_1',
    'close_lag_2',
    'close_lag_3',
    'close_lag_5',
    'close_lag_10',

    'ma_5',
    'ma_10',
    'ma_20',

    'return_1day',
    'return_5day',
    'return_10day',

    'volatility_5',
    'volatility_10',

    'price_range',
    'range_pct',

    'day',
    'day_of_week',
    'month',
    'quarter',

    'fund_encoded'
]

In [ ]:
target_high = 'target_high'
target_low = 'target_low'

In [ ]:
train = df[df['price_date'] < '2025-01-01']

validation = df[
    (df['price_date'] >= '2025-01-01') &
    (df['price_date'] < '2026-01-01')
]

test = df[df['price_date'] >= '2026-01-01']

In [ ]:
print(df['price_date'].min())
print(df['price_date'].max())

In [ ]:
X_train = train[features]
y_high_train = train['target_high']
y_low_train = train['target_low']

X_test = test[features]
y_high_test = test['target_high']
y_low_test = test['target_low']

In [ ]:
from sklearn.ensemble import RandomForestRegressor

high_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

low_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

In [54]:
high_model.fit(X_train, y_high_train)

KeyboardInterrupt: 

In [ ]:
low_model.fit(X_train, y_low_train)

In [ ]:
pred_high = high_model.predict(X_test)

In [ ]:
pred_low = low_model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error

high_mae = mean_absolute_error(
    y_high_test,
    pred_high
)

low_mae = mean_absolute_error(
    y_low_test,
    pred_low
)

print("High MAE:", high_mae)
print("Low MAE:", low_mae)

In [ ]:
from sklearn.metrics import root_mean_squared_error

high_rmse = root_mean_squared_error(
    y_high_test,
    pred_high
)

low_rmse = root_mean_squared_error(
    y_low_test,
    pred_low
)